We know from feature calculator and the linear regression the features which need to be considered (ladder_position, ladder_position_prev and percentage)

First check if algorithm works on ladder_Position_Next

In [ ]:
import pandas as pd

df = pd.read_csv("data/test.csv")

df["Year_Num"] = df["Year"].str.replace("x", "0").astype(int)
df["Percentage"] = df["Percentage"].fillna(100)
df["Ladder_Position"] = df["Ladder_Position"].fillna(df["Ladder_Position"].mean())
df["Ladder_Position_Prev"] = df["Ladder_Position_Prev"].fillna(df["Ladder_Position_Prev"].mean())

def predict_ladder_position_next_next(df):
    df = df.copy()

    required_cols = ["Ladder_Position", "Ladder_Position_Prev", "Percentage"]
    if not all(col in df.columns for col in required_cols):
        raise ValueError("Missing one or more required columns.")

    df["Change"] = df["Ladder_Position"] - df["Ladder_Position_Prev"]

    def adjust_change(row):
        percentage = row["Percentage"] / 100
        change = row["Change"]
        factor = percentage if percentage > 1 else (2 - percentage)
        return change * factor

    df["Adjusted_Change"] = df.apply(adjust_change, axis=1)

    df["Ladder_Position_Next_Next_Pred"] = df["Ladder_Position"] + df["Adjusted_Change"]

    return df[[
        "ID", "Year", "Ladder_Position", "Ladder_Position_Prev",
        "Percentage", "Ladder_Position_Next_Next_Pred"
    ]]

predicted_df = predict_ladder_position_next_next(df)

print(predicted_df[['Ladder_Position_Next_Next_Pred']])
print("Ladder position next: \n", df['Ladder_Position_Next'])
print("Avg off:", df['Ladder_Position_Next'].sum()/predicted_df[['Ladder_Position_Next_Next_Pred']].sum())


      Ladder_Position_Next_Next_Pred
0                           1.481600
1                           0.535200
2                          -9.943973
3                          12.000000
4                          -1.861400
...                              ...
1098                        8.258000
1099                        8.000000
1100                       14.476700
1101                       12.381200
1102                        0.589900

[1103 rows x 1 columns]
Ladder position next: 
 0        1
1       12
2        4
3       10
4        2
        ..
1098     7
1099     7
1100     3
1101    17
1102     7
Name: Ladder_Position_Next, Length: 1103, dtype: int64
Avg off: Ladder_Position_Next_Next_Pred    1.020578
dtype: float64


Lets predict for the season after

In [ ]:
df = pd.read_csv("data/test.csv")

df["Year_Num"] = df["Year"].str.replace("x", "0").astype(int)
df["Percentage"] = df["Percentage"].fillna(100)
df["Ladder_Position"] = df["Ladder_Position"].fillna(df["Ladder_Position"].mean())
df["Ladder_Position_Next"] = df["Ladder_Position_Next"].fillna(df["Ladder_Position_Next"].mean())

def predict_ladder_position_next_next(df):
    df = df.copy()

    required_cols = ["Ladder_Position", "Ladder_Position_Prev", "Percentage"]

    df["Change"] = df["Ladder_Position_Next"] - df["Ladder_Position"]

    def adjust_change(row):
        percentage = row["Percentage"] / 100
        change = row["Change"]
        factor = percentage if percentage > 1 else (2 - percentage)
        return change * factor

    df["Adjusted_Change"] = df.apply(adjust_change, axis=1)

    df["Ladder_Position_Next_Next_Pred"] = df["Ladder_Position_Next"] + df["Adjusted_Change"]

    return df[[
        "ID", "Year", "Ladder_Position", "Ladder_Position_Next",
        "Percentage", "Ladder_Position_Next_Next_Pred"
    ]]

predicted_df = predict_ladder_position_next_next(df)

print(predicted_df[['Ladder_Position_Next_Next_Pred']])
df = df.rename(columns={"Ladder_Position_Next_Next_Pred": "Ladder_Position_Next"})


predicted_df[["ID", "Ladder_Position_Next"]].to_csv("submission.csv", index=False)



      Ladder_Position_Next_Next_Pred
0                            -6.6048
1                            23.0916
2                             9.5143
3                             7.4152
4                             3.4307
...                              ...
1098                          9.1720
1099                          5.4875
1100                         -6.2712
1101                         26.8577
1102                         10.4101

[1103 rows x 1 columns]
